In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
!cp -r drive/MyDrive/PlantVillage /content/PlantVillage
path="/content/PlantVillage"

In [4]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import datasets, transforms
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import random


device = "cuda" if torch.cuda.is_available() else "cpu"


In [5]:
full_dataset = datasets.ImageFolder(root=path, transform=None)
targets = [sample[1] for sample in full_dataset.samples]
classes = list(full_dataset.class_to_idx.keys())

train_idx, test_idx = train_test_split(
    np.arange(len(targets)),
    test_size=0.2,
    random_state=42,
    stratify=targets
)

print(f"Dataset has {len(classes)} classes.")
print(f"Number of Train images: {len(train_idx)} | Number of Test images: {len(test_idx)}")


Dataset has 15 classes.
Number of Train images: 16567 | Number of Test images: 4142


In [6]:
temp_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

temp_full = datasets.ImageFolder(root=path, transform=temp_transform)
temp_train = Subset(temp_full, train_idx)

def get_mean_std(dataset, batch_size=32, num_workers=2, sample_size=2000):
    if sample_size and len(dataset) > sample_size:
        indices = random.sample(range(len(dataset)), sample_size)
        dataset = Subset(dataset, indices)

    loader = DataLoader(dataset, batch_size=batch_size, num_workers=num_workers, shuffle=False)

    channel_sum = 0.
    channel_sumsq = 0.
    count_pixels = 0

    for x, _ in tqdm(loader, desc="Calculating mean/std"):
        b, c, h, w = x.shape
        count_pixels += b * h * w
        channel_sum += x.sum()
        channel_sumsq += (x**2).sum()

    mean = channel_sum / count_pixels
    std = torch.sqrt(channel_sumsq / count_pixels - mean**2)
    return mean.item(), std.item()

mean, std = get_mean_std(temp_train, sample_size=3000)
print(f"Calculated Mean: {mean}")
print(f"Calculated Std: {std}")


Calculating mean/std:   0%|          | 0/94 [00:00<?, ?it/s]

Calculated Mean: 0.4634266197681427
Calculated Std: 0.16980648040771484


In [7]:
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean], std=[std])
])

test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean], std=[std])
])


In [8]:
train_data_all = datasets.ImageFolder(root=path, transform=train_transform)
test_data_all = datasets.ImageFolder(root=path, transform=test_transform)

train_set = Subset(train_data_all, train_idx)
test_set = Subset(test_data_all, test_idx)

BATCH_SIZE = 32
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of testing batches: {len(test_loader)}")


Number of training batches: 518
Number of testing batches: 130


In [9]:
class PlantVillageCNN(nn.Module):
    def __init__(self, output_shape):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv_block_3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=64 * 28 * 28, out_features=512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(in_features=512, out_features=output_shape)
        )

    def forward(self, x):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.conv_block_3(x)
        x = self.classifier(x)
        return x

model = PlantVillageCNN(output_shape=len(classes)).to(device)


In [10]:
def train_step(model, dataloader, loss_fn, optimizer):
    model.train()
    train_loss, train_acc = 0, 0
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)
    return train_loss / len(dataloader), train_acc / len(dataloader)

def test_step(model, dataloader, loss_fn):
    model.eval()
    test_loss, test_acc = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            y_pred = model(X)
            test_loss += loss_fn(y_pred, y).item()
            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
            test_acc += (y_pred_class == y).sum().item()/len(y_pred)
    return test_loss / len(dataloader), test_acc / len(dataloader)


In [ ]:
# Executed the below code for three times so it got trained for 45 epoches

In [13]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 15

for epoch in tqdm(range(EPOCHS)):
    train_loss, train_acc = train_step(model, train_loader, loss_fn, optimizer)
    test_loss, test_acc = test_step(model, test_loader, loss_fn)
    print(
        f"Epoch: {epoch+1} | "
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_acc*100:.2f}% | "
        f"Test loss: {test_loss:.4f} | "
        f"Test acc: {test_acc*100:.2f}%"
    )


  0%|          | 0/15 [00:00<?, ?it/s]

Epoch: 1 | Train loss: 0.3168 | Train acc: 89.44% | Test loss: 0.4358 | Test acc: 88.38%
Epoch: 2 | Train loss: 0.3141 | Train acc: 89.08% | Test loss: 0.4226 | Test acc: 88.22%
Epoch: 3 | Train loss: 0.3132 | Train acc: 89.50% | Test loss: 0.4084 | Test acc: 88.27%
Epoch: 4 | Train loss: 0.2927 | Train acc: 89.87% | Test loss: 0.4457 | Test acc: 88.07%
Epoch: 5 | Train loss: 0.3075 | Train acc: 89.67% | Test loss: 0.3928 | Test acc: 88.56%
Epoch: 6 | Train loss: 0.2918 | Train acc: 89.92% | Test loss: 0.4180 | Test acc: 88.85%
Epoch: 7 | Train loss: 0.2890 | Train acc: 90.51% | Test loss: 0.4291 | Test acc: 88.15%
Epoch: 8 | Train loss: 0.2733 | Train acc: 90.80% | Test loss: 0.4299 | Test acc: 88.85%
Epoch: 9 | Train loss: 0.2798 | Train acc: 90.36% | Test loss: 0.4273 | Test acc: 89.28%
Epoch: 10 | Train loss: 0.2691 | Train acc: 90.95% | Test loss: 0.4312 | Test acc: 89.10%
Epoch: 11 | Train loss: 0.2661 | Train acc: 90.77% | Test loss: 0.4894 | Test acc: 87.88%
Epoch: 12 | Train l

In [14]:
torch.save(model.state_dict(), "gray_model.pth")

In [17]:
model = PlantVillageCNN(len(classes))
model.load_state_dict(torch.load("gray_model.pth"))
model.to(device)
model.eval()

PlantVillageCNN(
  (conv_block_1): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_3): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=50176, out_features=512, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=512, out_features=15, bias=True)
  )
)

In [19]:
from google.colab import files
files.download("gray_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
from PIL import Image
from torchvision import transforms
import torch


In [24]:
image_path = "/content/PlantVillage/Tomato__Target_Spot/04889d08-6298-45d8-ab6c-0119fd1c2104___Com.G_TgS_FL 0778.JPG"

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean], std=[std])
])

img = Image.open(image_path)
img_tensor = transform(img).unsqueeze(0).to(device)


In [25]:
with torch.no_grad():
    outputs = model(img_tensor)
    predicted_class = torch.argmax(outputs, dim=1).item()

print(f"Predicted class: {classes[predicted_class]}")


Predicted class: Tomato__Target_Spot
